# EEG MMIDB exploration

Inspect preprocessed epochs from PhysioNet EEG Motor Movement/Imagery (`eegmmidb`).
Run `python3 data/download.py` and `python3 data/preprocess.py` first.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config
from models.utils import load_epochs

ds = load_epochs()
X, y, subjects = ds["X"], ds["y"], ds["subjects"]
print("X", X.shape, "y", y.shape, "subjects", len(np.unique(subjects)))
print("class counts", {n: int(np.sum(y == i)) for i, n in enumerate(config.CLASS_NAMES)})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = [np.sum(y == i) for i in range(len(config.CLASS_NAMES))]
ax.bar(config.CLASS_NAMES, counts, color="#2E86AB")
ax.set_ylabel("Epochs")
ax.set_title("Class distribution (imbalance is expected: rest is every trial)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
times = np.linspace(config.EPOCH_TMIN, config.EPOCH_TMAX, X.shape[-1])
fig, ax = plt.subplots(figsize=(8, 4))
for i, name in enumerate(config.CLASS_NAMES):
    grand = X[y == i].mean(axis=(0, 1))
    ax.plot(times, grand, label=name, lw=1.4)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV, grand average)")
ax.set_title("Grand-average ERP-like traces (all channels)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Motor imagery lives in **mu/beta (8–30 Hz)** over sensorimotor cortex, not in a large evoked spike.
The plot above is a sanity check that epochs are finite and class-conditional means differ slightly;
CSP / EEGNet operate on spatial covariance and spectro-spatial structure rather than this average.